In [1]:
%cd ~/cdv/

import os

# os.environ['CUDA_VISIBLE_DEVICES'] = ''

import numpy as np
import jax.numpy as jnp
import jax
import jax.random as jr
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import rho_plus as rp
import wat
from eins import EinsOp
import treescope


is_dark = False
theme, cs = rp.mpl_setup(is_dark)
rp.plotly_setup(is_dark)

/home/nmiklaucic/miniconda3/envs/avid/lib/python3.12/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/home/nmiklaucic/miniconda3/envs/avid/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/nmiklaucic/cdv


In [11]:
from pymatgen.core import Structure
from facet.config import MainConfig
from facet.checkpointing import best_ckpt
from pathlib import Path
import pyrallis

model_dir = Path('logs') / 'enb-198'

with open(model_dir / 'config.toml') as f:
    config = pyrallis.cfgparsing.load(MainConfig, f)

model = config.build_regressor()
ckpt = best_ckpt(model_dir)
# params = ckpt['state']['params']
params = ckpt['state']['opt_state'][-1]['ema']

In [4]:
from facet.data.dataset import load_file


cg = load_file(config, group_num=0, file_num=1)
cg

CrystalGraphs(nodes=(1024, 3), edges=(1024, 32), graphs=(32,))

In [12]:
from facet.layers import Context


model.apply(params, cg=cg, ctx=Context(training=False))


Array([[-8.497833 ],
       [-7.338007 ],
       [-6.2268405],
       [-7.427959 ],
       [-4.7085752],
       [-6.2136984],
       [-8.010998 ],
       [-7.3470287],
       [-7.6018305],
       [-3.823306 ],
       [-4.4961824],
       [-8.528979 ],
       [-7.011705 ],
       [-7.6672926],
       [-5.231403 ],
       [-7.6846366],
       [-6.677555 ],
       [-7.202202 ],
       [-2.3462543],
       [-5.5102954],
       [-4.832037 ],
       [-4.2712464],
       [-3.2748294],
       [-9.017158 ],
       [-7.379459 ],
       [-3.939594 ],
       [-7.4800916],
       [-2.1870835],
       [-4.5867977],
       [-5.8974   ],
       [-6.8834724],
       [-5.044202 ]], dtype=float32)

In [13]:
from facet.data.databatch import CrystalGraphs


CrystalGraphs?

Init signature:
CrystalGraphs(
    nodes: facet.data.databatch.NodeData,
    edges: facet.data.databatch.EdgeData,
    n_node: jaxtyping.Int[Array, 'graphs'],
    padding_mask: jaxtyping.Bool[Array, 'graphs'],
    graph_data: facet.data.databatch.CrystalData,
    target_data: facet.data.databatch.TargetInfo,
) -> None
Docstring:      Batched/padded graphs. Should be able to sub in for jraph.GraphsTuple.
File:           ~/cdv/facet/data/databatch.py
Type:           type
Subclasses:     

In [22]:
from pymatgen.core import Structure

struct = Structure.from_file('/home/nmiklaucic/ParetoCSP/results/Sr1Ti1O3_paretocsp_pop92/structures/Sr1Ti1O3_relaxed.cif')
struct


Structure Summary
Lattice
    abc : 2.95747991 3.3909 10.7788958
 angles : 90.0 90.0 90.0
 volume : 108.09635730684109
      A : 2.95747991 0.0 1.810934152662051e-16
      B : 5.452983092788506e-16 3.3909 2.0763274156143798e-16
      C : 0.0 0.0 10.7788958
    pbc : True True True
PeriodicSite: Sr0 (Sr) (1.479, 1.695, 8.068) [0.5, 0.5, 0.7485]
PeriodicSite: Ti1 (Ti) (1.479, 0.0, 0.6548) [0.5, 0.0, 0.06075]
PeriodicSite: O2 (O) (0.0, 0.0, 1.794) [0.0, 0.0, 0.1664]
PeriodicSite: O3 (O) (0.0, 0.0, 7.187) [0.0, 0.0, 0.6668]
PeriodicSite: O4 (O) (0.0, 0.0, 10.05) [0.0, 0.0, 0.9323]

In [92]:
import numpy as np
from vesin import NeighborList

rmax = params['params']['edge_embedding']['rmax'].item()

# positions can be anything compatible with numpy's ndarray
positions = struct.cart_coords
box = struct.lattice.matrix

calculator = NeighborList(cutoff=rmax, full_list=True)
P, S, d, D = calculator.compute(
    points=positions,
    box=box,
    periodic=True,
    quantities="PSdD"
)
d_sort = np.argsort(d)
P, S, d = P[d_sort], S[d_sort], d[d_sort]

In [97]:
from facet.data.databatch import NodeData, EdgeData, CrystalData, CrystalGraphs, TargetInfo

n_nodes = struct.num_sites
k = 32

z_to_i = {z: i for i, z in enumerate(config.data.metadata.atomic_numbers)}

to_jimage = np.zeros((n_nodes, k, 3), dtype=np.int32)
receiver = np.zeros((n_nodes, k), dtype=np.uint32)
species = np.zeros((n_nodes,), dtype=np.uint32)

for i in range(n_nodes):
    mask = np.argwhere(P[:, 0] == i).reshape(-1)
    mask = mask[:k]
    n = len(mask)
    to_jimage[[i], :n, :] = S[mask]
    receiver[i, :n] = P[mask, 1]
    species[i] = z_to_i[struct.species[i].Z]


cg_s = CrystalGraphs(
    NodeData(species, struct.cart_coords, np.zeros_like(species)),
    EdgeData(to_jimage, receiver),
    n_node = np.array([n_nodes]),
    padding_mask = np.array([1]),
    graph_data = CrystalData(
        dataset_id=np.array([0]),
        abc=np.array(struct.lattice.abc)[None, ...],
        angles_rad=np.deg2rad(struct.lattice.angles)[None, ...],
        lat=struct.lattice.matrix[None, ...]
    ),
    target_data=TargetInfo(np.array([0.]), np.zeros_like(struct.cart_coords), np.zeros((1, 3, 3)))
)

cg_s

CrystalGraphs(nodes=(5, 3), edges=(5, 32), graphs=(1,))

In [107]:
jax.jit(model.apply, static_argnames=['params', 'ctx'])(params, cg=cg_s, ctx=Context(training=False))

Array([[-6.1738863]], dtype=float32)